In [1]:
# %pip install python-dotenv
# %uv add dspy

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


### check aicodetools library

In [3]:
import time
import threading
import tiktoken
from collections import deque
import dspy
from dspy.utils.callback import BaseCallback


class SlidingWindowLimiter:
    """Rate limiter that enforces both request and token limits per rolling minute."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, max_requests_per_min=1000, max_tokens_per_min=2_000_000):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.max_requests = max_requests_per_min
            cls._instance.max_tokens = max_tokens_per_min
            cls._instance.requests = deque()  # [(timestamp, tokens)]
            cls._instance._lock = threading.Lock()
            cls._instance.encoder = tiktoken.get_encoding("cl100k_base")
        return cls._instance

    def _cleanup(self, now):
        """Remove entries older than 60s."""
        while self.requests and now - self.requests[0][0] > 60:
            self.requests.popleft()

    def _count(self):
        """Total requests & tokens in current 60s window."""
        total_tokens = sum(t for _, t in self.requests)
        return len(self.requests), total_tokens

    def acquire(self, tokens_used=0):
        """Wait until request fits in sliding 60s window."""
        with self._lock:
            while True:
                now = time.time()
                self._cleanup(now)
                req_count, token_count = self._count()

                # Can fit in current 60s window?
                if (req_count < self.max_requests and
                        token_count + tokens_used <= self.max_tokens):
                    # Record the new request
                    self.requests.append((now, tokens_used))
                    break  # proceed

                # Otherwise, figure out when we can retry
                oldest_time = self.requests[0][0]
                sleep_time = max(0.01, 60 - (now - oldest_time))
                print(f"⚠️ Throttling: sleeping {sleep_time:.2f}s (req={req_count}, tokens={token_count})")
                time.sleep(sleep_time)


class DelayAndLogCallback(BaseCallback):
    """DSPy callback using sliding window limiter."""

    def __init__(self):
        self.limiter = SlidingWindowLimiter()

    def _estimate_tokens(self, messages=None, prompt=None):
        """Estimate token usage using tiktoken."""
        text = ""
        if prompt:
            text = str(prompt)
        elif messages:
            # concatenate all message contents
            text = " ".join(m.get("content", "") for m in messages)
        return len(self.limiter.encoder.encode(text))

    def on_lm_start(self, *args, **kwargs):
        inputs = kwargs.get("inputs") or {}
        prompt = inputs.get("prompt")
        messages = inputs.get("messages")
        tokens_used = self._estimate_tokens(messages=messages, prompt=prompt)
        self.limiter.acquire(tokens_used=tokens_used)

    def on_lm_end(self, *args, **kwargs):
        pass


In [4]:

# from aicodetools import ClientManager 

# code_tool_manager = ClientManager(
#                 "super-bench:latest", base_log_dir="runs/super/"
#             )

# code_tool_client = code_tool_manager.get_client('initial')

In [5]:
import os
# os.environ['OPENAI_API_KEY'/] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [6]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=30, callbacks=[DelayAndLogCallback()])
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [7]:
# print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "t : Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [8]:

from gepa_artifact.benchmarks.super_bench.super_utils import FinishResponse
from gepa_artifact.benchmarks.super_bench import benchmark as sb_metas

Available Tools for ad325fec-f39c-460f-823f-e93bfbbe8a86: on runtime aicodetools-ad325fec-f39c-460f-823f-e93bfbbe8a86-f1f00e2f 4


In [9]:
bench = sb_metas[0].benchmark()

In [10]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(9, 9, 27)

In [ ]:
import pprint 

# x = copy paste ins froms state runs\gepa-state-with-gold\instruction_proposer_inpouts.jsonl

In [15]:
# pprint.pprint(x.keys())

# pprint.pprint(x['all_inputs'].keys())


# pprint.pprint(x['all_inputs']['user_examples_and_feedback'])

In [11]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'instance_id': 'pie-perf', 'github_repo': 'https://github.com/madaan/pie-perf', 'git_commit': 'ee1989b66756470622e3b89c4aa031f083f57ef9', 'query': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0). Once evaluated, report the result problem_id and input_acc for each problem of the dataset, as a json list of dictionaries structured as follows: [{"problem_id": "", "input_acc": 0.0}] (replace "" and 0.0 with the actual values).\n\nAdditional instructions:\n1. Set "num_trials": 2 in the evaluation configuration file to reduce computation time.\n2. Load only the first 10 rows of the dataset.\n\nGit repository: https://github.com/madaan/pie-perf', 'query_components': {'e2e_task': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0).', 'scenario_task': '

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [12]:
program = sb_metas[0].program[0]
program

react.react = Predict(StringSignature(query, github_repo, git_commit, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions='Solve the question and provide the answer in the correct format.\n\nYou are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.\nYour goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.\n\nTo do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.\nAfter each tool call, you receive a resulting observation, which gets appended to your trajectory.\n\nWhen writing next_thought, you may reason about the current situation and plan for future steps.\nWhen selecting the next_tool_name and its next_tool_args, the tool must be one of:\n\n(1) read_file, whose description is <desc>          Read file with optional line range and/or 

### Make Sure docker is installed and running

## Define an evaluator and evaluate the base program

In [28]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=sb_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='super_react.json',
    save_as_csv='super_react.csv'
)

In [ ]:
evaluate(program)

## on train and val set

In [ ]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.train_set+bench.val_set,
    metric=sb_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='super_react_train_val.json',
    save_as_csv='super_react_train_val.csv'
)

## Load the GEPA Optimizer

In [13]:
# Import GEPA and define the optimizer
from gepa_artifact.gepa.gepa import GEPA,GEPAState
from gepa_artifact.utils.capture_stream_logger import Logger

import time

runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if sb_metas[0].feedback_fn_maps is None or sb_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = sb_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = sb_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=sb_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    num_iters=9,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=9)

## Optimize the program with GEPA

In [14]:
sb_metas[0].program[0].get_lm()

## RUN to optimise the program

In [15]:
# optimized_program = optimizer.compile(
#     sb_metas[0].program[0],
#     trainset=bench.train_set,
#     valset=bench.val_set,
# )

In [16]:
# optimizer.gepa_state.save(runs_dir)

## Load from the Saved dir

In [17]:
state = GEPAState.load('runs/2025-10-20_03-45-26')

In [18]:
def idxmax(lst):
    """Return the index of the maximum value in a list."""
    max_val = max(lst)
    return lst.index(max_val)

In [24]:
gepa_state = state
best_prog_idx = idxmax(gepa_state.per_program_tracked_scores)
best_progs = gepa_state.program_candidates
best_prog = best_progs[best_prog_idx]

In [30]:
# best_progs
for idx,i in enumerate(best_progs):
    print(f'*************{idx}***************')
    for name, pred in i.named_predictors():
        print("================================")
        print(f"Predictor: {name}")
        print("================================")
        print("Prompt:")
        print(pred.signature.instructions)
        print("*********************************")

*************0***************
Predictor: react.react
Prompt:
Solve the question and provide the answer in the correct format.

You are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.
Your goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.

To do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.
After each tool call, you receive a resulting observation, which gets appended to your trajectory.

When writing next_thought, you may reason about the current situation and plan for future steps.
When selecting the next_tool_name and its next_tool_args, the tool must be one of:

(1) read_file, whose description is <desc>          Read file with optional line range and/or regex filtering.            Args:              file_path: Absolute path to file (e.g., /workspace

In [20]:
optimized_program=best_prog

### Let's print the prompts that GEPA discovered

In [21]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: react.react
Prompt:
You are an advanced ML experiment execution assistant focused on running controlled, reproducible machine learning benchmarks from academic repositories containing code, datasets, and configuration scripts.

Your tasks follow this recurring structure:
- You receive as input:
  - A `query` describing a controlled ML run with precise instructions on experiment, dataset restrictions, hyperparameter changes, and the needed format/result metric extraction (often a rubric-specified JSON dictionary).
  - The `github_repo` URL and `git_commit` specifying the codebase/commit to use.
  - Sometimes a `trajectory`—log of previous steps/actions/results for multi-step tasks.

Here are critical, domain-specific and general instructions for performing these experiments reliably and as intended:

**Repository and Directory Navigation**
- All tasks assume the codebase has been cloned at the specified commit; however, you must always verify correct repo state before attempt

## Now, let's evaluate the optimized program

In [22]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=sb_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='optimized_react.json',
    save_as_csv='optimized_react.csv'
)

In [ ]:
latest_program = best_progs[-1]

for name, pred in latest_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

## if we use the best program given by gepa

In [23]:
evaluate(optimized_program)

  0%|          | 0/27 [00:00<?, ?it/s]Available Tools for g-transformer: on runtime aicodetools-g-transformer-b8e70a6a 4
Available Tools for spa: on runtime aicodetools-spa-188aed72 4
Available Tools for mezo: on runtime aicodetools-mezo-3b8fb3c5 4
Available Tools for mode-connectivity-plm: on runtime aicodetools-mode-connectivity-plm-3eb02957 4
Available Tools for mbib: on runtime aicodetools-mbib-7b66bea8 4
Available Tools for unsupervisedhierarchicalsymbolicregression: on runtime aicodetools-unsupervisedhierarchicalsymbolicregression-0d6b09a7 4
Available Tools for conv_graph: on runtime aicodetools-conv_graph-13c4cf97 4
Available Tools for pira: on runtime aicodetools-pira-9867bb7d 4
Available Tools for pet: on runtime aicodetools-pet-c57174ea 4
Cleaned up Tools g-transformer : True  True
success=False structured_output={'Sentence-level BLEU': 0.0, 'Document-level BLEU': 0.0} reasoning='Step 1 (Repository Preparation): Inspection of the working directory showed that the requested re

2025/10/21 18:01:43 INFO dspy.evaluate.evaluate: Average Metric: 2.458333333333333 / 27 (9.1%)


,instance_id,github_repo,git_commit,query,query_components,answer,landmarks,trajectory,reasoning,result,super_score
0,g-transformer,https://github.com/baoguangsheng/g-transformer,dcc7695ceb0ecc3250e1c28215e9ddcd22700b39,Use the https://github.com/baoguangsheng/g-transformer repository ...,{'e2e_task': 'Use the https://github.com/baoguangsheng/g-transform...,"{""Sentence-level BLEU"": 0.0, ""Document-level BLEU"": 0.01}",['INFO\\] Building segmented data' 'INFO \\| fairseq_cli.preproces...,"{'thought_0': ""Before configuring or running the experiment, I nee...","Step 1: Repository Preparation Upon starting, I attempted to verif...","success=False structured_output={'Sentence-level BLEU': 0.0, 'Docu...","✔️ [Prediction(\n score=0.25,\n score_dict={'submitted': 1, ..."
1,spa,https://github.com/OceannTwT/SPA,a8bb190a182c076f80d90ac59921abd1df29b0ae,"Train the SPA model on alpaca_data_en_52k (from the repo), startin...",{'e2e_task': 'Train the SPA model on alpaca_data_en_52k (from the ...,"{""training_loss"": 4.97342586517334}","[Generating train split: \d+ examples, >> \*\*\*\*\* Running train...","{'thought_0': 'First, I need to check the contents of the current ...",Step 1: Repository Preparation The first protocol step requires ve...,success=False structured_output={'training_loss': 0.0} reasoning='...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
2,mezo,https://github.com/princeton-nlp/MeZO,552cb1b710767f9a6e1dc8f9645d7640376f9941,"Train using the ""MeZO"" method (no prefix-tuning / lora) on the RTE...","{'e2e_task': 'Train using the ""MeZO"" method (no prefix-tuning / lo...","{""accuracy"": 0.8, ""dev_accuracy"": 0.4}",['- INFO - Sample train set \\d+/\\d+' '- INFO - \\*\\*\\*\\*\\* R...,"{'thought_0': 'Before running the experiment, I need to verify tha...",Step 1: Repository Preparation Upon listing the contents of the wo...,"success=False structured_output={'accuracy': 0.0, 'dev_accuracy': ...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
3,mode-connectivity-plm,https://github.com/thunlp/mode-connectivity-plm,f45efd03fabdb23777c3ddcb35612cb1f7b1960c,"Fine-tune roberta-base on a single-task, ""rotten tomatoes"" which i...","{'e2e_task': 'Fine-tune roberta-base on a single-task, ""rotten tom...","{""eval_loss"": 0.0368}","[Running tokenizer on dataset:, \*\*\*\*\* train metrics \*\*\*\*\...","{'thought_0': ""Before proceeding with the experiment setup, I need...",Step 1 of the protocol requires verification of the repository at ...,"success=False structured_output={'eval_loss': 0.0} reasoning=""The ...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
4,mbib,https://github.com/Media-Bias-Group/MBIB,b9a887ffd461fa462e89835fc27b36e370091954,"Train a bart-base model on the ""linguistic-bias"" task using the ba...","{'e2e_task': 'Train a bart-base model on the ""linguistic-bias"" tas...","{""average_weighted_f1"": 0.44272727272727275}","[Training Initialized for fold \d+, The current dev loss: tensor\(...",{'thought_0': 'I need to begin by inspecting the current directory...,Stepwise reasoning and actions: 1. **Repository Preparation:** - I...,success=False structured_output={'average_weighted_f1': 0.0} reaso...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
5,unsupervisedhierarchicalsymbolicregression,https://github.com/SiyuLou/UnsupervisedHierarchicalSymbolicRegression,9fb1ba66ee1006f14c15f1b5b2390c8d0fa24fd6,Train a hierarchical symbolic regression model from this repo usin...,{'e2e_task': 'Train a hierarchical symbolic regression model from ...,"{""Rf_final"": {""R_square"": -0.25985264778137207, ""MSE"": 0.148261207...","[the size of train dataset is \d+, validation_dataset is \d+, test...","{'thought_0': ""To begin, I need to verify that I'm in the correct ...",Step 1: Repository Preparation I started by inspecting the /worksp...,"success=False structured_output={'Rf_final': {'R_square': 0.0, 'MS...","✔️ [Prediction(\n score=0.08333333333333333,\n

9.1

Available Tools for cet: on runtime aicodetools-cet-cd06f1c7 4
Available Tools for linkbert: on runtime aicodetools-linkbert-30faae73 4
Available Tools for team: on runtime aicodetools-team-1d1e52de 4
Cleaned up Tools cet : True  True
success=False structured_output={'best_dev_accuracy': 0.0, 'final_test_accuracy': 0.0} reasoning='Protocol terminated at Step 1: Repository Verification. The specified CET repository (https://github.com/zzz47zzz/CET, commit 8f3dabbf6fc6e90cfa0f6745ac99c4b23cd00833) is not present in the environment—the working directory is empty and contains no files or folders. This makes it impossible to proceed with any further steps: code inspection, dataset preparation, model fine-tuning, or accuracy evaluation. As per protocol, I must halt immediately, return requested output with placeholder metrics, and explain this early termination.' summary='The CET repository was missing from the environment so no experiment scripts, data, or methods were available. The proces

In [32]:
latest_program = best_progs[-1]

for name, pred in latest_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: react.react
Prompt:
You are an Agent designed to perform controlled machine learning evaluation experiments in code repositories and submit rubric-specific results. Your workflow must be strictly evidence-based: all actions, file selections, script modifications, and outputs must be directly grounded in file/directory content, in-script comments, README instructions, or results of commands you execute—never hypothesized beyond observable project artifacts.

Your end goal is to run evaluation experiments as specified by the task inputs, restricting data loading or experiment scope precisely (e.g., by category or row count), and extracting ONLY the output structure requested (such as a filtered JSON of scores or a dict of predictions) for submission. You must learn and apply domain- and repo-specific conventions as you go, adapting your logic based on the observable files/scripts/config/dataset organization.

Based on lessons and feedback from prior sessions, be aware of the f

In [ ]:
evaluate(latest_program)

  0%|          | 0/27 [00:00<?, ?it/s]Available Tools for g-transformer: on runtime aicodetools-g-transformer-4b71b9f0 4
Available Tools for spa: on runtime aicodetools-spa-39219a91 4
Available Tools for mezo: on runtime aicodetools-mezo-a6c956b2 4
Available Tools for mode-connectivity-plm: on runtime aicodetools-mode-connectivity-plm-3a07ec16 4
Available Tools for mbib: on runtime aicodetools-mbib-9a57a808 4
Available Tools for unsupervisedhierarchicalsymbolicregression: on runtime aicodetools-unsupervisedhierarchicalsymbolicregression-f51ef5a2 4
Available Tools for conv_graph: on runtime aicodetools-conv_graph-0bdd0925 4
Available Tools for pira: on runtime aicodetools-pira-a626529d 4
Available Tools for pet: on runtime aicodetools-pet-56874620 4
Cleaned up Tools g-transformer : True  True
success=False structured_output={'Sentence-level BLEU': None, 'Document-level BLEU': None} reasoning='The working directory is empty, and the git status indicates that no repository has been cloned

2025/10/21 18:59:46 WARNING dspy.utils.callback: Error when calling callback <__main__.DelayAndLogCallback object at 0x786bc32120c0>: deque index out of range


⚠️ Throttling: sleeping 0.01s (req=5, tokens=132889)
⚠️ Throttling: sleeping 0.06s (req=4, tokens=130340)
⚠️ Throttling: sleeping 0.04s (req=3, tokens=80324)
⚠️ Throttling: sleeping 0.03s (req=2, tokens=30308)
⚠️ Throttling: sleeping 0.01s (req=1, tokens=15154)
⚠️ Throttling: sleeping 58.59s (req=12, tokens=182490)
Available Tools for align-to-distill: on runtime aicodetools-align-to-distill-6e9f7984 4


2025/10/21 19:00:46 WARNING dspy.utils.callback: Error when calling callback <__main__.DelayAndLogCallback object at 0x786bc32120c0>: deque index out of range


⚠️ Throttling: sleeping 0.01s (req=1, tokens=50500)
Cleaned up Tools pira : True  True
success=False structured_output={'F1': None, 'accuracy': None} reasoning='Repeated attempts to clone and access the specified repository and commit revealed that it is completely empty. None of the C4AI/Pira answer triggering dataset, code, or scripts are present, so no experiment, training, or evaluation can be performed.' summary='Mapped and verified the requested commit of the Pira repository multiple times. No code, data, or instructions required for the answer triggering QA task are present in the repo at that commit. Cannot run experiment or report metrics. Strong evidence that files have been removed or the commit is an empty placeholder.'
The experiment could not be carried out due to repeated evidence that the specified repository and commit (`https://github.com/C4AI/Pira` at `4666d88f1ecec8c3662de3ffaa0d313d924529c2`) do not contain any of the required files for the answer triggering model 

...

Cleaned up Tools mbib : True  True
success=False structured_output={'average_weighted_f1': 'None'} reasoning="The experiment requires 'linguistic-bias.csv', but this file does not exist in the repository and cannot be created due to persistent shell/file I/O failures. Thus, it is impossible to restrict the input, configure, train, and evaluate bart-base as required. All preparatory steps and logic are complete pending file resolution." summary='Mapped the repo, determined required CSV format, synthetic data, and workflow for baseline bart-base training and weighted F1 evaluation. Blocked on dataset file creation due to technical/server errors; no results produced.'
To perform the requested experiment, I first mapped the repository and attempted to locate the "linguistic-bias.csv" file, its generation, or download source, but found it was not present and no generation instructions existed in scripts, configs, or documentation. Exhaustive exploration (including config and script inspecti

2025/10/21 19:48:46 INFO dspy.evaluate.evaluate: Average Metric: 1.375 / 27 (5.1%)


,instance_id,github_repo,git_commit,query,query_components,answer,landmarks,trajectory,reasoning,result,super_score
0,g-transformer,https://github.com/baoguangsheng/g-transformer,dcc7695ceb0ecc3250e1c28215e9ddcd22700b39,Use the https://github.com/baoguangsheng/g-transformer repository ...,{'e2e_task': 'Use the https://github.com/baoguangsheng/g-transform...,"{""Sentence-level BLEU"": 0.0, ""Document-level BLEU"": 0.01}",['INFO\\] Building segmented data' 'INFO \\| fairseq_cli.preproces...,"{'thought_0': ""My first step is to systematically map the reposito...","The repository directory is completely empty, with no files or fol...","success=False structured_output={'Sentence-level BLEU': None, 'Doc...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
1,spa,https://github.com/OceannTwT/SPA,a8bb190a182c076f80d90ac59921abd1df29b0ae,"Train the SPA model on alpaca_data_en_52k (from the repo), startin...",{'e2e_task': 'Train the SPA model on alpaca_data_en_52k (from the ...,"{""training_loss"": 4.97342586517334}","[Generating train split: \d+ examples, >> \*\*\*\*\* Running train...","{'thought_0': 'First, I need to map the repo contents to locate wh...",I attempted to locate the necessary files and directories within t...,success=False structured_output={'training_loss': 'None'} reasonin...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
2,mezo,https://github.com/princeton-nlp/MeZO,552cb1b710767f9a6e1dc8f9645d7640376f9941,"Train using the ""MeZO"" method (no prefix-tuning / lora) on the RTE...","{'e2e_task': 'Train using the ""MeZO"" method (no prefix-tuning / lo...","{""accuracy"": 0.8, ""dev_accuracy"": 0.4}",['- INFO - Sample train set \\d+/\\d+' '- INFO - \\*\\*\\*\\*\\* R...,"{'thought_0': 'First, I need to map the repository structure to id...",Despite repeated attempts to interact with the MeZO repository and...,"success=False structured_output={'accuracy': None, 'dev_accuracy':...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
3,mode-connectivity-plm,https://github.com/thunlp/mode-connectivity-plm,f45efd03fabdb23777c3ddcb35612cb1f7b1960c,"Fine-tune roberta-base on a single-task, ""rotten tomatoes"" which i...","{'e2e_task': 'Fine-tune roberta-base on a single-task, ""rotten tom...","{""eval_loss"": 0.0368}","[Running tokenizer on dataset:, \*\*\*\*\* train metrics \*\*\*\*\...","{'thought_0': 'First, I need to map the repository directory and l...",All attempts to access the file structure (using commands like `ls...,success=False structured_output={} reasoning='Multiple systematic ...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 0, '..."
4,mbib,https://github.com/Media-Bias-Group/MBIB,b9a887ffd461fa462e89835fc27b36e370091954,"Train a bart-base model on the ""linguistic-bias"" task using the ba...","{'e2e_task': 'Train a bart-base model on the ""linguistic-bias"" tas...","{""average_weighted_f1"": 0.44272727272727275}","[Training Initialized for fold \d+, The current dev loss: tensor\(...","{'thought_0': ""First, I need to map the repository directory struc...","To perform the requested experiment, I first mapped the repository...",success=False structured_output={'average_weighted_f1': 'None'} re...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
5,unsupervisedhierarchicalsymbolicregression,https://github.com/SiyuLou/UnsupervisedHierarchicalSymbolicRegression,9fb1ba66ee1006f14c15f1b5b2390c8d0fa24fd6,Train a hierarchical symbolic regression model from this repo usin...,{'e2e_task': 'Train a hierarchical symbolic regression model from ...,"{""Rf_final"": {""R_square"": -0.25985264778137207, ""MSE"": 0.148261207...","[the size of train dataset is \d+, validation_dataset is \d+, test...","{'thought_0': 'First, I need to map the repo structure to identify...","The repo was initially cloned successfully, and dependencies were ...","success=False structured_output={'Rf_final': {'R_square': 'None', ...","✔️ [Prediction(\n score=0.0,\n score_dict={'

5.09

2025/10/21 20:06:23 ERROR dspy.utils.parallelizer: Error for Example({'instance_id': 'mbib', 'github_repo': 'https://github.com/Media-Bias-Group/MBIB', 'git_commit': 'b9a887ffd461fa462e89835fc27b36e370091954', 'query': 'Train a bart-base model on the "linguistic-bias" task using the baseline scripts. Report the average weighted f1-score as a json structured as follows: {"average_weighted_f1": 0.0} (replace 0.0 with the actual value).\n\nAdditional instructions:\n1. From the generated `linguistic-bias.csv` file, take only the first and last 25 rows.\n2. Train only one epoch.\n\nGit repository: https://github.com/Media-Bias-Group/MBIB', 'query_components': {'e2e_task': 'Train a bart-base model on the "linguistic-bias" task using the baseline scripts.', 'scenario_task': '', 'report': 'Report the average weighted f1-score as a json structured as follows: {"average_weighted_f1": 0.0} (replace 0.0 with the actual value).', 'instructions': '1. From the generated `linguistic-bias.csv` file, ta